# 07 · RAG Knowledge Agent (retrieval as a tool)

**Where we are in the stack:** the **knowledge plane**. The loop from notebook 02 is unchanged;
the new capability is *looking things up*. RAG = **R**etrieval-**A**ugmented **G**eneration.

The model's weights are like a router's factory firmware: frozen at training time. Your
runbooks, your device quirks, last week's incident review - none of that is in there. RAG is
the **route lookup** for knowledge: instead of memorising every prefix (fine-tuning), you keep
a table (a vector index) and do a longest-prefix match (nearest-neighbour search) at query time.

```
question -> embed -> nearest chunks -> stuff into context -> model answers, citing them
```

We hand-roll the vector store in ~15 lines (numpy only) so you can see there is no magic,
then plug retrieval into the **same agent loop** as one more tool: `search_docs`.

> Needs: a tool-capable model (see notebook 02) **and** an embeddings endpoint. On OpenAI the
> default `text-embedding-3-small` just works; on Ollama set `EMBED_MODEL=nomic-embed-text`.

In [ ]:
# --- Provider config: works with OpenAI, OpenRouter, or a local OpenAI-compatible server ---
import os
from openai import OpenAI

# Load settings from a .env file if present (falls back to existing env vars).
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    import os
    if os.path.exists(".env"):
        for _line in open(".env"):
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())


# Pick ONE setup by exporting these env vars before launching Jupyter.
#
#   OpenAI:     OPENAI_BASE_URL=https://api.openai.com/v1   MODEL=gpt-4o-mini
#   OpenRouter: OPENAI_BASE_URL=https://openrouter.ai/api/v1 MODEL=openai/gpt-4o-mini
#   Local:      OPENAI_BASE_URL=http://localhost:11434/v1    MODEL=qwen2.5:7b   (Ollama)
#               (use 'qwen2.5' / 'llama3.1' etc. - a 1B model is great for chat but
#                usually too weak to drive tool-calling reliably.)

BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
API_KEY  = os.environ.get("OPENAI_API_KEY", "set-me")   # any non-empty string for local servers
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

# Behind a TLS-intercepting firewall/proxy, HTTPS cert verification can fail.
# Set VERIFY_SSL=false in .env to skip it: we hand the OpenAI SDK a custom
# httpx client with verification turned off. Leave it true everywhere else.
import httpx
VERIFY_SSL = os.environ.get("VERIFY_SSL", "true").strip().lower() not in ("false", "0", "no")
http_client = httpx.Client(verify=VERIFY_SSL)
if not VERIFY_SSL:
    import warnings
    warnings.filterwarnings("ignore")
    print("\u26a0\ufe0f  SSL verification DISABLED (VERIFY_SSL=false) \u2014 use only on a trusted network")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, http_client=http_client)
print("endpoint:", BASE_URL, "| model:", MODEL)

## 1. Knowledge the model cannot have

We generate a few **fictional internal runbooks** into `runbooks/` (git-ignored, recreated on
each run). The facts in them are made up, so the model *cannot* answer from its weights - if
it answers correctly, retrieval worked. Swap in your own `.md` files to index real docs.

In [ ]:
import os, textwrap

DOCS_DIR = "runbooks"
os.makedirs(DOCS_DIR, exist_ok=True)

RUNBOOKS = {
    "rb-101-crc-errors.md": """
        # RB-101: CRC errors on 10G leaf uplinks
        Symptom: incrementing CRC errors on ethernet1/0/x 10G ports on AlmondOS leaf switches.
        Root cause: NIC firmware bug FRM-88 in firmware 4.1.x mis-frames jumbo packets.
        Fix: upgrade leaf NIC firmware to 4.2.1 or later during a maintenance window.
        Workaround: cap MTU at 1500 on the affected link until the upgrade.
        Owner: datacenter-net team, escalation alias net-tier2.
    """,
    "rb-207-bgp-flaps.md": """
        # RB-207: BGP session flaps between spine and edge
        Symptom: BGP sessions to the edge routers flap every 90-120 seconds.
        Root cause history: 9 out of 10 past incidents were an MTU mismatch on the
        transit VLAN (edge expects 9000, spine default is 1500), which drops large
        UPDATE messages while keepalives still pass.
        Fix: set MTU 9000 on the spine-side transit interfaces, then clear the session.
        Never restart the edge router for this - it does not help and drops customer traffic.
    """,
    "rb-330-maintenance-policy.md": """
        # RB-330: Maintenance window policy
        Standard windows: Tuesday and Thursday 02:00-05:00 UTC.
        Emergency changes outside a window need approval from the on-call network duty
        manager AND an open SEV-1 or SEV-2 incident ticket.
        Firmware upgrades on leaf switches take ~25 minutes per device and must be done
        one switch per rack at a time to keep ECMP capacity.
    """,
}

for fname, body in RUNBOOKS.items():
    with open(os.path.join(DOCS_DIR, fname), "w") as f:
        f.write(textwrap.dedent(body).strip() + "\n")

print("wrote", len(RUNBOOKS), "runbooks into", DOCS_DIR + "/")

## 2. Embeddings: text → vector

An embedding maps text to a point in space where **similar meaning = nearby points**.
"CRC errors on the uplink" and "checksum failures on the 10G port" share almost no words,
but their vectors sit close together - that is why this beats `grep` for questions.

We chunk each runbook by paragraph (embedding whole files blurs them; single lines lose
context), embed every chunk once, and L2-normalise so **cosine similarity is just a dot
product**. Same `client`, same `.env` - only the endpoint (`/embeddings`) is new.

In [ ]:
import numpy as np

EMBED_MODEL = os.environ.get("EMBED_MODEL", "text-embedding-3-small")

def embed(texts):
    """Embed a list of strings -> (n, d) unit-length numpy matrix."""
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = np.array([d.embedding for d in resp.data], dtype=np.float32)
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)   # unit length

# Chunk = one paragraph, tagged with its source file so answers can cite it.
chunks, sources = [], []
for fname in sorted(os.listdir(DOCS_DIR)):
    text = open(os.path.join(DOCS_DIR, fname)).read()
    for para in text.split("\n\n"):
        para = para.strip()
        if para:
            chunks.append(para)
            sources.append(fname)

INDEX = embed(chunks)     # the entire "vector database": one numpy matrix
print(f"indexed {len(chunks)} chunks from {len(set(sources))} files, dim={INDEX.shape[1]}")

## 3. A vector store in ~15 lines

A "vector database" at demo scale is a matrix and an argsort. (Real ones add persistence,
approximate search for millions of vectors, filtering, and updates - the FIB to our
routing-table-in-RAM.) Sanity-check it **without the model first**, same as we did for
tools in notebook 02.

In [ ]:
def search_docs(query, k=3):
    """Return the k most relevant runbook chunks for a query, with scores + sources."""
    k = max(1, min(int(k), len(chunks)))
    qv = embed([query])[0]
    scores = INDEX @ qv                       # cosine similarity via dot product
    top = np.argsort(-scores)[:k]
    return {"results": [
        {"source": sources[i], "score": round(float(scores[i]), 3), "text": chunks[i]}
        for i in top
    ]}

# No LLM involved yet - retrieval is deterministic given the index.
for r in search_docs("checksum failures on a ten-gig port", k=2)["results"]:
    print(f"{r['score']:.3f}  {r['source']}  {r['text'][:70]}...")

## 4. Retrieval as a *tool* - same loop, new capability table

Two common RAG shapes:
- **Pipeline RAG**: always retrieve, staple the chunks to the prompt, answer once.
- **Agentic RAG** (this notebook): retrieval is a **tool** the model calls when it wants,
  as many times as it wants, rephrasing the query between calls.

The loop below is the FSM from notebook 02 - only `TOOLS`, `TOOL_REGISTRY`
and the system prompt changed. That invariance is still the lesson.

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "search_docs",
        "description": "Semantic search over the internal network runbooks. "
                       "Returns the most relevant chunks with source filenames.",
        "parameters": {"type": "object",
            "properties": {
                "query": {"type": "string", "description": "natural-language search query"},
                "k":     {"type": "integer", "description": "how many chunks (default 3)"}},
            "required": ["query"]}}},
]

TOOL_REGISTRY = {"search_docs": search_docs}

### The agent loop (the same FSM as notebook 2)

In [ ]:
import json

SYSTEM = """You are a network-operations knowledge assistant.
Answer ONLY from the internal runbooks, which you access via the search_docs tool.
Strategy:
  1. Search before you answer; rephrase and search again if the first hits are weak.
  2. Ground every claim in a retrieved chunk and cite its source file, e.g. (rb-101-crc-errors.md).
  3. If the runbooks do not cover it, say so - do not fall back to general knowledge."""

def ask_docs(question, max_iterations=6, temperature=0):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    print("Q:", question)
    print("=" * 72)

    for step in range(1, max_iterations + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, temperature=temperature,
        )
        msg = resp.choices[0].message

        # TERMINATION: no tool requested -> final answer.
        if not msg.tool_calls:
            print(f"\n[step {step}] FINAL ANSWER\n{'-'*72}\n{msg.content}")
            return msg.content

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            print(f"[step {step}] TOOL  -> {name}({args})")
            try:
                result = TOOL_REGISTRY[name](**args)         # <-- WE run it
            except Exception as e:
                result = {"error": str(e)}
            preview = str(result)
            print(f"[step {step}] RESULT <- {preview[:200]}{'...' if len(preview) > 200 else ''}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result)[:6000],
            })

    return "Stopped: hit max_iterations (TTL expired)."

## 5. Run it - a fact that exists only in the runbooks

Firmware bug "FRM-88" is fictional. A correct, cited answer can only come from retrieval.

In [ ]:
_ = ask_docs("We are seeing CRC errors on a 10G leaf uplink. What is the root cause and the fix?")

## 6. Run it - a question that needs two different runbooks

Watch the agent search more than once: the fix comes from one document, the
scheduling constraints from another.

In [ ]:
_ = ask_docs(
    "BGP to the edge keeps flapping. What should we do, and when are we "
    "allowed to make the change if it is not an emergency?"
)

## Recap

You added a **knowledge plane** to the same agent:

- **Embeddings** turn text into vectors where similar meaning is nearby - semantic `grep`.
- The "vector database" was a numpy matrix and an argsort; real ones add scale and persistence.
- **Retrieval became a tool** (`search_docs`) in the same capability table, in the same loop -
  the model decides when and what to search, and cites what it saw.
- RAG beats fine-tuning for facts that change: update the *documents*, not the *weights* -
  edit a routing table instead of reflashing firmware.

Sharp edges to know about: chunking strategy matters, retrieval can miss (garbage in,
confident garbage out), and retrieved text is *untrusted input* - notebook 12 comes back
to that with guardrails.

**Next:** tools that live outside your process, discovered over a standard protocol. ->
`08_mcp_tool_server.ipynb`